# M4 — Oolel LLM-generated external baseline

This notebook downloads a pinned snapshot of `soynade-research/Wolof-Non-Standard-Orthography`, audits it, removes `<NON_STANDARD>` wrappers, converts accepted rows to the common benchmark schema, checks exact overlap, and creates a 100-row manual-review sheet.

M4 is an **external-data condition**: its 3,438 source sentences and generation procedure are not automatically controlled against M0/M1. It must be labelled accordingly in the results.

## Step 1 — Environment

If an import is missing, run the commented installation command once, restart the kernel if requested, and rerun the notebook. The Hugging Face download is performed only when `DOWNLOAD_DATASET=True`.

In [1]:
# %pip install -U datasets pyarrow rapidfuzz

import json
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('Run this notebook from the project root or notebooks directory')
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    GOLD_ANNOTATIONS_PATH, M4_OOLEL_MANIFEST_PATH, M4_OOLEL_PATH,
    M4_OOLEL_RAW_METADATA_PATH, M4_OOLEL_RAW_PATH,
    M4_OOLEL_REJECTED_PATH, M4_OOLEL_REVIEW_PATH, TRAIN_PARQUET_PATH,
)
from src.generation.benchmark_baselines import load_formal_sources, sha256_file
from src.generation.oolel_baseline import (
    NONSTANDARD_COLUMN_CANDIDATES, OOLEL_DATASET_ID, OOLEL_LICENSE,
    OOLEL_REVISION, add_overlap_flags, prepare_oolel, rejection_counts,
    resolve_nonstandard_column, write_m4_outputs,
)

## Step 2 — Frozen configuration

In [2]:
DOWNLOAD_DATASET = True  # Change to True for the first run.
OVERWRITE_OUTPUTS = True  # Set False once the benchmark is frozen.
EXPECTED_CARD_ROWS = 3438
REVIEW_SIZE = 100
REVIEW_SEED = 2026

print(f'Dataset:  {OOLEL_DATASET_ID}')
print(f'Revision: {OOLEL_REVISION}')
print(f'License:  {OOLEL_LICENSE}')
print(f'Raw path: {M4_OOLEL_RAW_PATH}')

Dataset:  soynade-research/Wolof-Non-Standard-Orthography
Revision: ce17c80cff6626a91dc4e38b43de12e005bc240a
License:  CC BY-SA 4.0
Raw path: C:\Users\thioy\Desktop\School\PFE\data\external\oolel_raw.parquet


## Step 3 — Download or load the pinned raw snapshot

On the first run, set `DOWNLOAD_DATASET=True`. Later runs load the saved Parquet snapshot. The metadata sidecar prevents a local file from silently being attributed to another revision.

In [3]:
raw = None
if DOWNLOAD_DATASET:
    from datasets import load_dataset

    dataset = load_dataset(
        OOLEL_DATASET_ID, revision=OOLEL_REVISION, split='train'
    )
    raw = dataset.to_pandas()
    M4_OOLEL_RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
    raw.to_parquet(M4_OOLEL_RAW_PATH, index=False)
    raw_metadata = {
        'dataset_id': OOLEL_DATASET_ID,
        'revision': OOLEL_REVISION,
        'license': OOLEL_LICENSE,
        'rows': len(raw),
        'columns': list(raw.columns),
        'raw_sha256': sha256_file(M4_OOLEL_RAW_PATH),
    }
    M4_OOLEL_RAW_METADATA_PATH.write_text(
        json.dumps(raw_metadata, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    print(f'Downloaded and saved {len(raw):,} rows.')
elif M4_OOLEL_RAW_PATH.exists() and M4_OOLEL_RAW_METADATA_PATH.exists():
    raw_metadata = json.loads(M4_OOLEL_RAW_METADATA_PATH.read_text(encoding='utf-8'))
    if raw_metadata.get('revision') != OOLEL_REVISION:
        raise ValueError('Local Oolel snapshot has a different revision. Redownload it.')
    if raw_metadata.get('raw_sha256') != sha256_file(M4_OOLEL_RAW_PATH):
        raise ValueError('Local Oolel raw checksum does not match its metadata.')
    raw = pd.read_parquet(M4_OOLEL_RAW_PATH)
    print(f'Loaded local pinned snapshot with {len(raw):,} rows.')
else:
    print('No local snapshot. Set DOWNLOAD_DATASET=True and rerun this cell.')

Generating train split:   0%|          | 0/3438 [00:00<?, ? examples/s]

Downloaded and saved 3,438 rows.


## Step 4 — Verify source schema

The dataset metadata currently exposes `non_standardized`, while the prose card calls it `non_standard`. The preparation code accepts both but records the field actually found.

In [4]:
if raw is None:
    raise RuntimeError('Download the pinned dataset in Step 3 before continuing.')

nonstandard_column = resolve_nonstandard_column(raw.columns)
assert 'wo' in raw.columns
assert 'en' in raw.columns
if len(raw) != EXPECTED_CARD_ROWS:
    print(f'WARNING: expected {EXPECTED_CARD_ROWS:,} rows from the card, received {len(raw):,}.')
print(f'Columns: {list(raw.columns)}')
print(f'Using non-standard field: {nonstandard_column!r}')
raw.head()

Columns: ['en', 'wo', 'non_standardized']
Using non-standard field: 'non_standardized'


,en,wo,non_standardized
0,"Constance: My cat, Sangeetha, is always there ...","Constance: Sama muus, Sangeetha daf may buyu s...","<NON_STANDARD>\nConstance: Sama mousse, Sangee..."
1,[The people continue to chant.],[Nit ñi wéy di way.],<NON_STANDARD>\nNit gni way di wayé\n</NON_STA...
2,"Joseph: Oh no, Jasmine.","Joseph: Déet waay, Jasmine.",<NON_STANDARD>\nJoseph: Déet waye Jasmine\n</N...
3,"You'll feel refreshed and rejuvenated, ready t...","Beesaat naa tàkk boole dellu ndaw, mën naa tàm...",<NON_STANDARD>\nBessate na takk bolé dellou nd...
4,But for every person who found success in the ...,"Waaye su amee kenn ku tekki ci dëkk, amna ñene...",<NON_STANDARD>\nWayé sou amé kene kou tekki si...


## Step 5 — Clean wrappers and create the common M4 schema

In [5]:
m4, audit = prepare_oolel(raw, revision=OOLEL_REVISION)

assert m4.source_id.is_unique
assert m4.formal_wolof.str.len().gt(0).all()
assert m4.informal_wolof.str.len().gt(0).all()
assert ~m4.informal_wolof.str.contains('<NON_STANDARD', case=False, regex=False).any()

print(f'Raw rows:      {len(audit):,}')
print(f'Accepted rows: {len(m4):,}')
print(f'Rejected rows: {(~audit.accepted).sum():,}')
print(f'Changed rows:  {m4.changed.sum():,}')
print(f'Rejections:    {dict(rejection_counts(audit))}')
display(audit.wrapper_status.value_counts(dropna=False))
m4.head()

Raw rows:      3,438
Accepted rows: 3,433
Rejected rows: 5
Changed rows:  3,433
Rejections:    {'duplicate_pair': 5}


wrapper_status
valid    3438
Name: count, dtype: int64

,source_id,source_row,source_corpus,method,seed,informal_wolof,formal_wolof,english,changed,edit_count,error_types,applied_rules
0,oolel:0,0,soynade-research/Wolof-Non-Standard-Orthograph...,m4_oolel_llm_external,not_reported,"Constance: Sama mousse, Sangeetha dafmaye buyo...","Constance: Sama muus, Sangeetha daf may buyu s...","Constance: My cat, Sangeetha, is always there ...",True,22,"[""external_llm_generation_unspecified""]",{}
1,oolel:1,1,soynade-research/Wolof-Non-Standard-Orthograph...,m4_oolel_llm_external,not_reported,Nit gni way di wayé,[Nit ñi wéy di way.],[The people continue to chant.],True,6,"[""external_llm_generation_unspecified""]",{}
2,oolel:2,2,soynade-research/Wolof-Non-Standard-Orthograph...,m4_oolel_llm_external,not_reported,Joseph: Déet waye Jasmine,"Joseph: Déet waay, Jasmine.","Joseph: Oh no, Jasmine.",True,3,"[""external_llm_generation_unspecified""]",{}
3,oolel:3,3,soynade-research/Wolof-Non-Standard-Orthograph...,m4_oolel_llm_external,not_reported,"Bessate na takk bolé dellou ndaw, meun na tamb...","Beesaat naa tàkk boole dellu ndaw, mën naa tàm...","You'll feel refreshed and rejuvenated, ready t...",True,13,"[""external_llm_generation_unspecified""]",{}
4,oolel:4,4,soynade-research/Wolof-Non-Standard-Orthograph...,m4_oolel_llm_external,not_reported,Wayé sou amé kene kou tekki si deuk amna gnéné...,"Waaye su amee kenn ku tekki ci dëkk, amna ñene...",But for every person who found success in the ...,True,40,"[""external_llm_generation_unspecified""]",{}


## Step 6 — Exact overlap audit

This is a diagnostic only: gold text is not used to edit or select M4 outputs. Repeat the gold checks after train/dev/test are locked, then exclude any test overlap from training.

In [6]:
common_sources = load_formal_sources(TRAIN_PARQUET_PATH)
gold_formal = None
gold_informal = None
if GOLD_ANNOTATIONS_PATH.exists():
    annotations = pd.read_csv(GOLD_ANNOTATIONS_PATH)
    annotations = annotations.drop_duplicates('source_index', keep='last')
    gold_formal = annotations.get('manual_formal_wolof')
    gold_informal = annotations.get('comment')

audit = add_overlap_flags(
    audit, common_formal=common_sources.formal_wolof,
    gold_formal=gold_formal, gold_informal=gold_informal,
)
print(f"Formal overlap with 17,777-row common pool: {audit.overlap_common_formal.sum():,}")
print(f"Overlap with current gold formal text:       {audit.overlap_current_gold_formal.sum():,}")
print(f"Overlap with current gold informal text:     {audit.overlap_current_gold_informal.sum():,}")

Formal overlap with 17,777-row common pool: 3,438
Overlap with current gold formal text:       0
Overlap with current gold informal text:     0


## Step 7 — Intrinsic diagnostics and examples

In [7]:
print(m4.edit_count.describe())
print(f'Duplicate formal sources: {m4.formal_wolof.duplicated().sum():,}')
print(f'Duplicate informal strings: {m4.informal_wolof.duplicated().sum():,}')
m4[['formal_wolof', 'informal_wolof', 'edit_count']].sample(
    min(20, len(m4)), random_state=REVIEW_SEED
)

count    3433.000000
mean       20.172153
std         9.916159
min         1.000000
25%        13.000000
50%        19.000000
75%        26.000000
max        89.000000
Name: edit_count, dtype: float64
Duplicate formal sources: 3
Duplicate informal strings: 0


,formal_wolof,informal_wolof,edit_count
808,"Cafka yàpp, lujum yi ak fromaas yiñ ci boole d...","Thiafka yap, lidioum yi ak fromag yi gnou si b...",43
985,Way wu neex la am kàddu yu rafet.,Way bou nekh la am khaddou you rafet.,8
578,"Mu jàngal ko fi ñuy tàmbalee ci nataal suuf, w...",Moungui diangal ko fi gnuy tambalé ci nataal s...,25
615,Dafa yendoo jaay alumet waaye amul kenn ku jënd.,Dafa yendo djay alumét wayé amoul kene kou die...,13
736,Ñépp a wara xeex ngir sañ-sañu fësal xibaar.,Gneup a wara khekh nguir sagn sagnou feusal kh...,23
3099,Marse luuma yi ci dëkk yi ak ville barina lu f...,Marssé louma yi thi deuk yi ak ville barina lo...,12
2350,"Man tekkikatu làkk laa, li may tekki taalif yi...",Mane teukikatou lak la li may teuki talif yi t...,26
650,"Lily xamni dañu wara dimbali mbonaat bi, mu da...","Lily khamni daniou wara dimbeuli mbonate bi, m...",18
1026,Waaye li waraloon metitu bopp bii bisu liggéey...,Wayé li waralone metitou bopp bi bisou liguey ...,15
294,"Tekki làkk lu am solo la, lu mëna jàppale ab l...","Tekki lakk lou am solo la, lou meuna diappalé ...",19


## Step 8 — Export M4, rejected rows, review sheet, and manifest

The accepted CSV is the training condition. Rejected rows remain available for audit. The Excel-friendly review CSV uses UTF-8 with BOM.

In [8]:
output_paths = [
    M4_OOLEL_PATH, M4_OOLEL_REJECTED_PATH,
    M4_OOLEL_REVIEW_PATH, M4_OOLEL_MANIFEST_PATH,
]
if not OVERWRITE_OUTPUTS and any(path.exists() for path in output_paths):
    raise FileExistsError('M4 outputs already exist. Enable overwrite intentionally.')

manifest = write_m4_outputs(
    m4, audit, raw_path=M4_OOLEL_RAW_PATH, output_path=M4_OOLEL_PATH,
    rejected_path=M4_OOLEL_REJECTED_PATH, review_path=M4_OOLEL_REVIEW_PATH,
    manifest_path=M4_OOLEL_MANIFEST_PATH, review_size=REVIEW_SIZE,
    review_seed=REVIEW_SEED,
)
manifest

{'method': 'm4_oolel_llm_external',
 'created_at_utc': '2026-08-03T16:36:21.260460+00:00',
 'dataset_id': 'soynade-research/Wolof-Non-Standard-Orthography',
 'revision': 'ce17c80cff6626a91dc4e38b43de12e005bc240a',
 'license': 'CC BY-SA 4.0',
 'raw_path': 'C:\\Users\\thioy\\Desktop\\School\\PFE\\data\\external\\oolel_raw.parquet',
 'raw_sha256': 'f55bdee0e7769e19b987fa4e7eb5bffad422b8ef2184f87749b1f09e3ac06d39',
 'output_path': 'C:\\Users\\thioy\\Desktop\\School\\PFE\\data\\synthetic\\m4_oolel_external_pairs.csv',
 'output_sha256': 'ef435f83795d70833cad728a7867c13c649d6d4f927f42ecbc12a075d36ce351',
 'raw_rows': 3438,
 'accepted_rows': 3433,
 'rejected_rows': 5,
 'changed_rows': 3433,
 'rejection_counts': {'duplicate_pair': 5},
 'wrapper_status_counts': {'valid': 3438},
 'exact_common_formal_overlap': 3438,
 'exact_current_gold_formal_overlap': 0,
 'exact_current_gold_informal_overlap': 0,
 'manual_review_rows': 100,
 'manual_review_seed': 2026,
 'limitations': ['External formal source c

## Step 9 — Manual work and benchmark use

1. Open `data/synthetic/m4_oolel_manual_review_sample.csv`.
2. Review all 100 rows for meaning preservation, informal plausibility, natural code-switching, and synthetic artifacts.
3. Do **not** silently correct reviewed rows. Report the quality rate; define a general rejection rule only if it can be applied without seeing gold test data.
4. Repeat overlap checks after the gold split is locked and exclude any overlap from model training.
5. Train M4 as an external-data condition. For size-controlled experiments, sample M0/M1 and later methods to the final accepted M4 row count with the same fixed seed.
6. Record that Oolel's original generation seed, prompt, decoding parameters, and row-level edit provenance are not reported in the dataset.